# Assignment 3 — MNIST GAN
Run from this project folder after installing the training dependencies. The shared modules contain the required architecture and training loop. The API uses the same saved generator. Standard GAN BCE replaces the CelebA WGAN loss to match the assignment.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from app.helper_lib.model import get_model
from app.helper_lib.trainer import train_gan
from app.helper_lib.generator import generate_samples

device = (torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cuda") if torch.cuda.is_available()
          else torch.device("cpu"))
print("Using device:", device)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize([0.5], [0.5])
])
dataset = datasets.MNIST("./data", train=True, download=True, transform=transform)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=0)
model = get_model("GAN").to(device)
print(model)

In [ ]:
criterion = nn.BCELoss()
optimizer = {
    "generator": optim.RMSprop(model.generator.parameters(), lr=5e-5),
    "discriminator": optim.RMSprop(model.discriminator.parameters(), lr=5e-5),
}
model = train_gan(model, dataloader, criterion, optimizer, device, epochs=20)
generate_samples(model, device, num_samples=10)

In [ ]:
model.generator.to("cpu")
torch.save(model.generator.state_dict(), "app/mnist_generator.pth")